# Chapter 3 Demo: Machine Learning and the Train-Test Split

Train a model, test it on unseen data, and see how it can fail.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-03-machine-learning.ipynb)

Lecture: [en-03-machine-learning](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-03-machine-learning.md)  
Labs: [LM301](https://rathachai.github.io/DA-LAB/learnings/linear/lm301.html) | [LM302](https://rathachai.github.io/DA-LAB/learnings/linear/lm302.html)

Three short add-ons follow: (A) split ratio, (B) noise features, (C) data leakage.

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---

## Step 1 · Load libraries
We use **pandas** for tables, **numpy** for maths, **scikit-learn** for the model and scores, and **matplotlib** only for plots.

เราใช้ **pandas** สำหรับตาราง, **numpy** สำหรับการคำนวณ, **scikit-learn** สำหรับโมเดลและคะแนน และใช้ **matplotlib** สำหรับการวาดกราฟเท่านั้น

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error,
                             mean_absolute_percentage_error, r2_score)

print("Libraries ready")

## Step 2 · Get data
`lm302_data.csv` has 100 rows: seven input features `x1..x7`, a measured answer `y`, and a `split` column saying **train** or **test**.

`lm302_data.csv` มี 100 แถว: ฟีเจอร์อินพุตเจ็ดตัว `x1..x7`, คำตอบที่วัดได้ `y` และคอลัมน์ `split` ที่บอกว่าเป็น **train** หรือ **test**

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm302_data.csv"
df = pd.read_csv(url)
print("rows, columns:", df.shape)

## Step 3 · Display data
Always look first: a few rows, the size, summary numbers, and how many rows are train vs test.

ควรดูข้อมูลก่อนเสมอ: ดูไม่กี่แถว ขนาดข้อมูล ค่าสรุปทางสถิติ และจำนวนแถวชุดฝึก (train) เทียบกับชุดทดสอบ (test)

In [ ]:
print(df.shape)

In [ ]:
df

In [ ]:
print(df["split"].value_counts())

In [ ]:
df.describe().round(2)

## Step 4 · Visualize correlation
Correlation **r** says how strongly a column moves together with `y` (+1 or -1 = strong, 0 = unrelated). Tall bars are useful features; short bars look like noise.

ค่าสหสัมพันธ์ **r** บอกว่าคอลัมน์หนึ่งเคลื่อนที่ไปพร้อมกับ `y` มากเพียงใด (+1 หรือ -1 = แรง, 0 = ไม่เกี่ยวข้อง) แท่งสูงคือฟีเจอร์ที่มีประโยชน์ แท่งเตี้ยดูเหมือนสัญญาณรบกวน (noise)

In [ ]:
r = df.drop(columns=["id", "split", "y"]).corrwith(df["y"])
r.plot.bar(figsize=(6, 3.5), rot=0)
plt.ylabel("r with y"); plt.title("Correlation of each column with y")

In [ ]:
print(r.round(2).to_dict())

## Step 5 · Process data
Feature `x5` curves with `y` (big jumps at small values), so a logarithm straightens it. We add a new column `ln_x5`.

ฟีเจอร์ `x5` โค้งเมื่อเทียบกับ `y` (กระโดดมากที่ค่าน้อย) ลอการิทึมจึงช่วยทำให้เป็นเส้นตรงขึ้น เราเพิ่มคอลัมน์ใหม่ `ln_x5`

In [ ]:
df["ln_x5"] = np.log(df["x5"])
print("r(x5, y) =", round(df["x5"].corr(df["y"]), 2), "  r(ln_x5, y) =", round(df["ln_x5"].corr(df["y"]), 2))

## Step 6 · Select X and y
`X` = the inputs the model may look at (the good features). `y` = the answer we want to predict. `x3` and `x4` look like noise, so we leave them out for now.

`X` = อินพุตที่โมเดลดูได้ (ฟีเจอร์ที่ดี) `y` = คำตอบที่ต้องการทำนาย `x3` และ `x4` ดูเหมือนสัญญาณรบกวน จึงยังไม่ใช้ในตอนนี้

In [ ]:
X = df[["x1", "x2", "ln_x5", "x6", "x7"]]
y = df["y"]
print(X.shape, y.shape)

In [ ]:
X

## Step 7 · Split into train and test
The model learns from **train** rows only. The **test** rows are kept hidden, like an exam the student has never seen. The file's `split` column is a random 70/30 split already made for you.

โมเดลเรียนรู้จากแถว **train** เท่านั้น แถว **test** ถูกซ่อนไว้ เหมือนข้อสอบที่นักเรียนไม่เคยเห็น คอลัมน์ `split` ในไฟล์เป็นการสุ่มแบ่ง 70/30 ที่ทำไว้ให้แล้ว

In [ ]:
is_train = df["split"] == "train"
X_train, X_test = X[is_train], X[~is_train]
y_train, y_test = y[is_train], y[~is_train]
print("train:", len(X_train), " test:", len(X_test))

The same idea done by scikit-learn yourself: `train_test_split` with `test_size=0.3`. `random_state=42` makes the shuffle repeatable. The rows differ from the file's split, but the idea is identical. (We keep the file's split for the rest.)

แนวคิดเดียวกันที่ทำด้วย scikit-learn เอง: `train_test_split` ด้วย `test_size=0.3` โดย `random_state=42` ทำให้การสุ่มทำซ้ำได้ แถวที่ได้จะต่างจากการแบ่งในไฟล์ แต่แนวคิดเหมือนกัน (ส่วนที่เหลือเราใช้การแบ่งตามไฟล์)

In [ ]:
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=42)
print("train:", len(Xa), " test:", len(Xb))

## Step 8 · Create and train the model
Like calibrating an instrument: we feed it known examples (`fit`) and it finds the best weights.

เหมือนการสอบเทียบเครื่องมือวัด: เราป้อนตัวอย่างที่ทราบคำตอบให้ (`fit`) แล้วโมเดลจะหาค่าน้ำหนักที่ดีที่สุด

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
print(pd.Series(model.coef_, index=X.columns).round(3))

In [ ]:
print("intercept:", round(model.intercept_, 3))

## Step 9 · Predict
Ask the trained model for answers on the train rows and on the unseen test rows.

ให้โมเดลที่ฝึกแล้วทำนายคำตอบของแถว train และแถว test ที่ยังไม่เคยเห็น

In [ ]:
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)
pd.DataFrame({"actual": y_test.values, "predicted": y_pred}).head().round(2)

## Step 10 · Evaluate
Four scores: **MAE** (average miss, in y units), **RMSE** (like MAE but punishes big misses), **MAPE** (average miss in %), **R²** (1 = perfect, 0 = no better than guessing the average). A model that studied well scores similarly on train and test. A **big gap** (train great, test poor) means it memorised instead of understood (overfitting).

คะแนนสี่แบบ: **MAE** (ความคลาดเคลื่อนเฉลี่ย หน่วยเดียวกับ y), **RMSE** (คล้าย MAE แต่ลงโทษความคลาดเคลื่อนใหญ่มากกว่า), **MAPE** (ความคลาดเคลื่อนเฉลี่ยเป็น %), **R²** (1 = สมบูรณ์แบบ, 0 = ไม่ดีกว่าการเดาค่าเฉลี่ย) โมเดลที่เรียนรู้ดีจะได้คะแนนใกล้เคียงกันทั้ง train และ test ส่วน**ช่องว่างที่กว้าง** (train ดีมากแต่ test แย่) หมายถึงโมเดลท่องจำแทนที่จะเข้าใจ (overfitting)

In [ ]:
mae = mean_absolute_error(y_train, y_train_pred)
print("train MAE  =", round(mae, 3))

rmse = root_mean_squared_error(y_train, y_train_pred)
print("train RMSE =", round(rmse, 3))

mape = 100 * mean_absolute_percentage_error(y_train, y_train_pred)
print("train MAPE =", round(mape, 2), "%")

r2 = r2_score(y_train, y_train_pred)
print("train R2   =", round(r2, 3))

mae = mean_absolute_error(y_test, y_pred)
print("test  MAE  =", round(mae, 3))

rmse = root_mean_squared_error(y_test, y_pred)
print("test  RMSE =", round(rmse, 3))

mape = 100 * mean_absolute_percentage_error(y_test, y_pred)
print("test  MAPE =", round(mape, 2), "%")

r2 = r2_score(y_test, y_pred)
print("test  R2   =", round(r2, 3))

## Step 11 · Visualize y and y_pred
Actual vs predicted. Points on the diagonal line are perfect predictions. **Circles = train, diamonds = test.**

ค่าจริงเทียบกับค่าที่ทำนาย จุดบนเส้นทแยงมุมคือการทำนายที่สมบูรณ์แบบ **วงกลม = train, สี่เหลี่ยมข้าวหลามตัด = test**

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_train, y_train_pred, marker="o", label="train")
plt.scatter(y_test, y_pred, marker="D", label="test")
lim = [y.min(), y.max()]
plt.plot(lim, lim, "k--", label="y = y_pred")
plt.xlabel("actual y"); plt.ylabel("predicted y"); plt.legend()

## (A) Does the split ratio matter?
For train shares from 10 % to 90 %, make 30 random splits and record the test RMSE. One split is a noisy estimate, so we look at the **mean ± spread (std)**.

สำหรับสัดส่วน train ตั้งแต่ 10 % ถึง 90 % สุ่มแบ่ง 30 ครั้งและบันทึกค่า RMSE ของ test การแบ่งครั้งเดียวเป็นการประมาณที่มีสัญญาณรบกวน เราจึงดู**ค่าเฉลี่ย ± ส่วนเบี่ยงเบน (std)**

In [ ]:
rows = []
for pct in range(10, 100, 10):
    for seed in range(30):
        Xa, Xb, ya, yb = train_test_split(X, y, train_size=pct / 100, random_state=seed)
        m = LinearRegression().fit(Xa, ya)
        rows.append({"train %": pct, "test_rmse": root_mean_squared_error(yb, m.predict(Xb))})
tab = pd.DataFrame(rows).groupby("train %")["test_rmse"].agg(["mean", "std"])
tab.round(3)

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.errorbar(tab.index, tab["mean"], yerr=tab["std"], fmt="o-", capsize=4)
plt.xlabel("train share (%)"); plt.ylabel("test RMSE"); plt.title("mean ± std over 30 splits")

Small train share: the model learns little, so error is high. Large train share: the test set is tiny, so the score is noisy. Both widen the error bars.

สัดส่วน train น้อย: โมเดลเรียนรู้น้อย ความคลาดเคลื่อนจึงสูง สัดส่วน train มาก: ชุด test เล็กมาก คะแนนจึงแกว่ง ทั้งสองกรณีทำให้แถบความคลาดเคลื่อนกว้างขึ้น

## (B) Noise features and overfitting
Now add the noise features `x3` and `x4` to X. They carry no real signal, but a model with spare dials can use them to memorise. Compare train vs test MAE at a small (20 %) and a large (80 %) train share, 30 splits each.

ตอนนี้เพิ่มฟีเจอร์รบกวน `x3` และ `x4` เข้าใน X ฟีเจอร์เหล่านี้ไม่มีสัญญาณจริง แต่โมเดลที่มีตัวปรับเหลือเฟือสามารถใช้มันเพื่อท่องจำได้ เปรียบเทียบ MAE ของ train กับ test ที่สัดส่วน train น้อย (20 %) และมาก (80 %) อย่างละ 30 การแบ่ง

In [ ]:
sets = {"good": ["x1", "x2", "ln_x5", "x6", "x7"],
        "good + noise": ["x1", "x2", "ln_x5", "x6", "x7", "x3", "x4"]}
rows = []
for pct in (20, 80):
    for seed in range(30):
        tr, te = train_test_split(df, train_size=pct / 100, random_state=seed)
        for name, f in sets.items():
            m = LinearRegression().fit(tr[f], tr["y"])
            rows.append({"train %": pct, "features": name,
                         "train MAE": mean_absolute_error(tr["y"], m.predict(tr[f])),
                         "test MAE": mean_absolute_error(te["y"], m.predict(te[f]))})
cmp_ = pd.DataFrame(rows).groupby(["train %", "features"]).mean()
cmp_["gap"] = cmp_["test MAE"] - cmp_["train MAE"]
cmp_.round(3)

In [ ]:
cmp_[["train MAE", "test MAE"]].plot.bar(figsize=(7, 3.5), rot=15)
plt.ylabel("mean MAE"); plt.title("Noise features: train vs test")

With noise features, train MAE drops a little (the model fits noise) while test MAE does not improve or gets worse. The effect is stronger with only 20 % of rows for training.

เมื่อมีฟีเจอร์รบกวน MAE ของ train ลดลงเล็กน้อย (โมเดลเรียนรู้ noise) แต่ MAE ของ test ไม่ดีขึ้นหรือแย่ลง ผลนี้ชัดกว่าเมื่อใช้แถวเพียง 20 % สำหรับการฝึก

## (C) Data leakage
Leakage = the test rows secretly helped build the model. Classic case: **feature selection**. We simulate 100 samples, 200 pure-noise features and a noise `y`, so honest R² should be about 0 or below.

**A (leak):** pick the 5 best features using ALL rows, then split. **B (honest):** split first, pick features from train rows only.

การรั่วไหล (leakage) = แถว test แอบมีส่วนช่วยสร้างโมเดล ตัวอย่างคลาสสิกคือ **การเลือกฟีเจอร์ (feature selection)** เราจำลองข้อมูล 100 ตัวอย่าง ฟีเจอร์ noise ล้วน 200 ตัว และ `y` ที่เป็น noise ดังนั้น R² ที่ซื่อสัตย์ควรอยู่ราว 0 หรือต่ำกว่า

**A (รั่วไหล):** เลือก 5 ฟีเจอร์ที่ดีที่สุดโดยใช้แถวทั้งหมด แล้วจึงแบ่งข้อมูล **B (ซื่อสัตย์):** แบ่งข้อมูลก่อน แล้วเลือกฟีเจอร์จากแถว train เท่านั้น

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
rng = np.random.default_rng(0)
Xs = pd.DataFrame(rng.normal(size=(100, 200)), columns=[f"f{i}" for i in range(200)])
ys = pd.Series(rng.normal(size=100))
rows = []
for seed in range(30):
    cols_a = Xs.columns[SelectKBest(f_regression, k=5).fit(Xs, ys).get_support()]
    Xa, Xb, ya, yb = train_test_split(Xs, ys, train_size=0.5, random_state=seed)
    cols_b = Xs.columns[SelectKBest(f_regression, k=5).fit(Xa, ya).get_support()]
    model_a = LinearRegression().fit(Xa[cols_a], ya)
    model_b = LinearRegression().fit(Xa[cols_b], ya)
    rows.append({"A: all rows (leak)": r2_score(yb, model_a.predict(Xb[cols_a])),
                 "B: train rows only": r2_score(yb, model_b.predict(Xb[cols_b]))})
lk = pd.DataFrame(rows)
lk.describe().round(3)

In [ ]:
lk.plot.box(figsize=(6, 3.5))
plt.axhline(0, color="gray", ls="--"); plt.ylabel("test R2")

A looks better only because the selection peeked at the test rows. B is the honest answer: no skill. The same rule applies to scalers and any pre-processing: fit on train only.

An even worse leak: give the model the answer `y` as a feature.

A ดูดีกว่าเพราะการเลือกฟีเจอร์แอบดูแถว test เท่านั้น B คือคำตอบที่ซื่อสัตย์: ไม่มีความสามารถในการทำนาย กฎเดียวกันใช้กับตัวปรับสเกล (scaler) และการประมวลผลข้อมูลล่วงหน้าทุกแบบ: fit จาก train เท่านั้น

การรั่วไหลที่แย่กว่านั้นอีก: ให้คำตอบ `y` แก่โมเดลเป็นฟีเจอร์

In [ ]:
f = sets["good + noise"]
a, b = train_test_split(df, train_size=0.2, random_state=0)
with_y = LinearRegression().fit(a[f + ["y"]], a["y"])
without = LinearRegression().fit(a[f], a["y"])
print("test R2 with y as a feature:", round(r2_score(b["y"], with_y.predict(b[f + ["y"]])), 4))

In [ ]:
print("test R2 without it         :", round(r2_score(b["y"], without.predict(b[f])), 4))

## Summary

- Judge a model on **unseen test data**; train error is optimistic, and a big train-test gap means overfitting.
- One split is noisy: repeat it and report mean and spread. Noise features hurt most when training data is small.
- **Leakage** (peeking at test rows, or using `y` as a feature) gives scores that look great and mean nothing. A test R² of 1.0 is a red flag.

- ประเมินโมเดลจาก**ข้อมูล test ที่ไม่เคยเห็น**; ความคลาดเคลื่อนของ train มองโลกในแง่ดีเกินจริง และช่องว่าง train-test ที่กว้างหมายถึง overfitting
- การแบ่งครั้งเดียวมี noise: ทำซ้ำแล้วรายงานค่าเฉลี่ยและการกระจาย ฟีเจอร์ noise ส่งผลเสียมากที่สุดเมื่อข้อมูลฝึกมีน้อย
- **การรั่วไหล (leakage)** (แอบดูแถว test หรือใช้ `y` เป็นฟีเจอร์) ทำให้คะแนนดูดีแต่ไม่มีความหมาย R² ของ test เท่ากับ 1.0 เป็นสัญญาณอันตราย

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---